# Telco Customer Churn Prediction

An end-to-end classification project using the IBM-style **Telco Customer Churn** dataset published by `blastchar` on Kaggle. The notebook cleans the data, explores churn patterns, preprocesses mixed data types, compares Logistic Regression with Random Forest, and explains the strongest predictors.

**Reproducibility:** fixed random seed (`42`), stratified 80/20 split, and preprocessing fitted only on the training set to prevent leakage.


## 1. Setup and load the CSV

In Google Colab, upload `WA_Fn-UseC_-Telco-Customer-Churn.csv` with the Files sidebar. If the file is missing, the code opens Colab's uploader automatically.


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f'Place {DATA_PATH} in the notebook folder.')

raw = pd.read_csv(DATA_PATH)
print('Raw shape:', raw.shape)
raw.head()


## 2. Data quality and cleaning

`TotalCharges` contains blank strings and therefore loads as text. Coercing it to numeric reveals 11 missing values. These records all have zero tenure, so they are newly created accounts with no accumulated charges; we remove them because 11 rows are only 0.16% of the dataset.


In [ ]:
print(raw.info())
print('Duplicate rows:', raw.duplicated().sum())
print('Duplicate customer IDs:', raw['customerID'].duplicated().sum())

df = raw.copy()
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('Missing TotalCharges after conversion:', df['TotalCharges'].isna().sum())
display(df.loc[df['TotalCharges'].isna(), ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head())

df = df.dropna(subset=['TotalCharges']).copy()
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})
print('Clean shape:', df.shape)


## 3. Exploratory analysis

Validated full-dataset findings:

- Overall churn: **26.58%**
- Month-to-month churn: **42.71%**, versus **11.28%** for one-year and **2.85%** for two-year contracts
- Churn falls from **47.68%** in months 0–12 to **9.51%** in months 49–72


In [ ]:
overall_churn = df['Churn'].mean() * 100
contract_churn = df.groupby('Contract')['Churn'].mean().mul(100).sort_values(ascending=False)

df['tenure_group'] = pd.cut(
    df['tenure'], bins=[-1, 12, 24, 48, 72],
    labels=['0-12 months', '13-24 months', '25-48 months', '49-72 months']
)
tenure_churn = df.groupby('tenure_group', observed=False)['Churn'].mean().mul(100)

print(f'Overall churn: {overall_churn:.2f}%')
display(contract_churn.rename('Churn rate (%)').to_frame())
display(tenure_churn.rename('Churn rate (%)').to_frame())

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
sns.countplot(data=df, x='Churn', ax=axes[0], palette='Blues')
axes[0].set_title('Customer churn counts')
axes[0].set_xticklabels(['Stayed', 'Churned'])
contract_churn.plot(kind='bar', ax=axes[1], color='#4C78A8')
axes[1].set_title('Churn rate by contract')
axes[1].set_ylabel('Churn rate (%)')
axes[1].tick_params(axis='x', rotation=25)
tenure_churn.plot(kind='bar', ax=axes[2], color='#F58518')
axes[2].set_title('Churn rate by tenure')
axes[2].set_ylabel('Churn rate (%)')
axes[2].tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()


## 4. Prepare the data

`customerID` is an identifier rather than a behavior signal, so it is excluded. Numeric values are median-imputed and standardized; categorical values are mode-imputed and one-hot encoded. The target is stratified so the train/test sets retain the original churn ratio.


In [ ]:
df = df.drop(columns='tenure_group')
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']

categorical = X.select_dtypes(include='object').columns.tolist()
numeric = X.select_dtypes(exclude='object').columns.tolist()

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric),
    ('cat', categorical_pipe, categorical)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print('Train:', X_train.shape, 'Test:', X_test.shape)


## 5. Train and evaluate two models

Accuracy alone can be misleading because only 26.6% of customers churn. We therefore compare precision, recall, F1, and ROC-AUC as well.


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(
        n_estimators=500, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
    )
}

fitted = {}
rows = []
predictions = {}
for name, estimator in models.items():
    pipe = Pipeline([('preprocess', preprocessor), ('model', estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    report = classification_report(y_test, pred, output_dict=True, zero_division=0)
    rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision (churn)': report['1']['precision'],
        'Recall (churn)': report['1']['recall'],
        'F1 (churn)': report['1']['f1-score'],
        'ROC-AUC': roc_auc_score(y_test, proba)
    })
    fitted[name] = pipe
    predictions[name] = (pred, proba)

metrics = pd.DataFrame(rows).set_index('Model')
display(metrics.style.format('{:.3f}'))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, (pred, _)) in zip(axes, predictions.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=['Stayed', 'Churned'], ax=ax, cmap='Blues')
    ax.set_title(name)
plt.tight_layout()
plt.show()


### Verified holdout results (seed 42)

| Model | Accuracy | Churn precision | Churn recall | Churn F1 | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| Logistic Regression | **80.38%** | **64.85%** | **57.22%** | **60.80%** | **0.836** |
| Random Forest | 78.89% | 63.14% | 49.47% | 55.47% | 0.826 |

Logistic Regression wins on every reported holdout metric. Its ROC-AUC of 0.836 indicates good ranking ability, though recall shows that a default 0.50 threshold still misses some churners. In a retention campaign, the threshold should be tuned to the relative cost of outreach versus a lost customer.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, (_, proba) in predictions.items():
    RocCurveDisplay.from_predictions(y_test, proba, name=name, ax=ax)
ax.plot([0, 1], [0, 1], '--', color='gray')
ax.set_title('ROC curves on the holdout set')
plt.show()


## 6. Feature importance

Random Forest importance is used as a directional explanation, not proof of causality. Continuous variables can receive more importance opportunities than individual one-hot indicators, so related category levels should be read together.


In [ ]:
rf = fitted['Random Forest']
feature_names = rf.named_steps['preprocess'].get_feature_names_out()
importance = pd.Series(
    rf.named_steps['model'].feature_importances_, index=feature_names
).sort_values(ascending=False).head(15)

importance.index = (importance.index
                    .str.replace('num__', '', regex=False)
                    .str.replace('cat__', '', regex=False))
display(importance.rename('importance').to_frame())

plt.figure(figsize=(9, 6))
sns.barplot(x=importance.values, y=importance.index, color='#4C78A8')
plt.title('Top 15 Random Forest feature importances')
plt.xlabel('Importance')
plt.ylabel('')
plt.tight_layout()
plt.show()


## 7. Business conclusions

1. **Contract commitment is the clearest churn segment.** Month-to-month customers churn at 42.71%, nearly 15× the two-year rate (2.85%). Offer contract-upgrade incentives without assuming the contract itself causes retention.
2. **The first year is the critical intervention window.** Customers with 0–12 months of tenure churn at 47.68%; strengthen onboarding and trigger proactive support early.
3. **Billing and service variables add useful signal.** Total charges, monthly charges, missing online security/tech support, fiber service, and electronic check appear among the strongest Random Forest features.
4. **Use probability scores operationally.** Logistic Regression performed best overall. Tune the decision threshold and evaluate lift/recall at the outreach team's capacity before deployment.

**Limitations:** This is observational, historical data. Feature importance is not causal, no intervention economics are provided, and performance should be revalidated on newer customers before production use.
